ARTI308 - Machine Learning

# Credit Card Customer Segmentation Project

In this project, you will use K-Means clustering to segment [credit card customers](https://www.kaggle.com/datasets/arjunbhasin2013/ccdata/data) based on their usage behavior. This is an unsupervised learning problem because the dataset does not contain a target label for customer groups.

You will use the `CC_GENERAL.csv` dataset.

## About the Dataset

The dataset contains customer-level credit card usage behavior. Each row represents one credit card holder, and the columns describe different behavioral variables such as balance, purchases, cash advance, payments, and tenure. The goal is to group similar customers together so that the company can understand different customer segments and design better marketing strategies.

## Import Libraries

**Import the libraries you need for data analysis, visualization, preprocessing, clustering, and evaluation.**

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.preprocessing import StandardScaler


from sklearn.cluster import KMeans


from sklearn.metrics import silhouette_score


from sklearn.decomposition import PCA


import warnings
warnings.filterwarnings('ignore')


sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## Get the Data

**Read the `CC_GENERAL.csv` file and save it in a dataframe called `df`.**

In [ ]:
df = pd.read_csv('CC_GENERAL.csv')

**Check the first five rows of the dataset.**

In [ ]:
df.head()

**Check the shape of the dataset.**

In [ ]:
print(f"Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns")

**Check basic information about the dataset using `info()`.**

In [ ]:
df.info()

**Check summary statistics using `describe()`.**

In [ ]:
df.describe()

## Data Cleaning

The column `CUST_ID` is an identification column. It is not useful for clustering because it does not describe customer behavior.

**Drop the `CUST_ID` column from the dataframe.**

In [ ]:
df.drop(columns=['CUST_ID'], inplace=True)
print("CUST_ID column dropped.")
print(f"New shape: {df.shape}")

**Check the missing values in each column.**

In [ ]:
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0])

Some columns may contain missing values.

Hint: You can handle missing values by either:
- filling them with the mean value
- or dropping the rows that contain missing values

For this project, use mean imputation.

**Fill the missing values with the mean of each column.**

In [ ]:
df.fillna(df.mean(), inplace=True)
print("Missing values filled with column mean.")

**Check the missing values again to make sure they were handled.**

In [ ]:
print("Missing values after imputation:")
print(df.isnull().sum().sum(), "total missing values")

## Exploratory Data Analysis

Before applying clustering, it is important to understand the data.

**Create histograms for the numerical columns.**

In [ ]:
df.hist(bins=30, figsize=(18, 14), color='steelblue', edgecolor='white')
plt.suptitle('Distribution of All Features', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**Create a correlation heatmap to understand relationships between the features.**

In [ ]:
plt.figure(figsize=(14, 10))
corr_matrix = df.corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, annot_kws={'size': 7})
plt.title('Correlation Heatmap of Credit Card Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Create a scatter plot between `BALANCE` and `PURCHASES`.**

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(df['BALANCE'], df['PURCHASES'], alpha=0.3, color='steelblue', edgecolors='none', s=15)
plt.xlabel('Balance', fontsize=12)
plt.ylabel('Purchases', fontsize=12)
plt.title('Balance vs Purchases', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Create a scatter plot between `BALANCE` and `CASH_ADVANCE`.**

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(df['BALANCE'], df['CASH_ADVANCE'], alpha=0.3, color='darkorange', edgecolors='none', s=15)
plt.xlabel('Balance', fontsize=12)
plt.ylabel('Cash Advance', fontsize=12)
plt.title('Balance vs Cash Advance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Feature Scaling

K-Means is a distance-based algorithm. Therefore, feature scaling is very important.

The features in this dataset have very different ranges. For example, `BALANCE`, `PURCHASES`, and `CREDIT_LIMIT` may have large values, while frequency columns are between 0 and 1.

**Use StandardScaler to scale the data. Save the scaled data in a variable called `X_scaled`.**

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)
print(f"Scaled data shape: {X_scaled.shape}")

## Choosing K Intuitively

Choosing K is one of the most difficult parts of K-Means.

Since this dataset has many features, it is not easy to visually see the clusters directly.

However, we can still compare different K values using the elbow method and silhouette score.

## Elbow Method

**Create a loop that fits K-Means models for K values from 1 to 10. Save the inertia values in a list called `inertia_values`.**

In [ ]:
inertia_values = []

for k in range(1, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia_values.append(kmeans.inertia_)

print("Inertia values for K = 1 to 10:")
for k, inertia in enumerate(inertia_values, start=1):
    print(f"  K={k}: {inertia:.2f}")

**Plot the elbow curve.**

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(range(1, 11), inertia_values, marker='o', linewidth=2,
         markersize=8, color='steelblue')
plt.xlabel('Number of Clusters (K)', fontsize=12)
plt.ylabel('Inertia (WCSS)', fontsize=12)
plt.title('Elbow Method — Optimal K Selection', fontsize=14, fontweight='bold')
plt.xticks(range(1, 11))
plt.tight_layout()
plt.show()

**Output Interpretation**

Look at the elbow curve and try to identify where the decrease in inertia starts to slow down.

That point can suggest a reasonable value for K.

## Silhouette Score

The silhouette score helps evaluate how well-separated the clusters are.

**Create a loop that calculates the silhouette score for K values from 2 to 10. Save the scores in a list called `silhouette_scores`.**

In [ ]:
silhouette_scores = []

for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)

**Plot the silhouette scores.**

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(range(2, 11), silhouette_scores, marker='s', linewidth=2,
         markersize=8, color='darkorange')
plt.xlabel('Number of Clusters (K)', fontsize=12)
plt.ylabel('Silhouette Score', fontsize=12)
plt.title('Silhouette Score for Each K Value', fontsize=14, fontweight='bold')
plt.xticks(range(2, 11))
plt.tight_layout()
plt.show()

**Create a table showing each K value and its silhouette score.**

In [ ]:
silhouette_df = pd.DataFrame({
    'K (Number of Clusters)': range(2, 11),
    'Silhouette Score': [round(s, 4) for s in silhouette_scores]
})
silhouette_df = silhouette_df.set_index('K (Number of Clusters)')
print(silhouette_df.to_string())

**Output Interpretation**

A higher silhouette score usually means better clustering.

However, do not rely only on the highest value. Also consider whether the chosen K makes sense for customer segmentation.

## Create the Final K-Means Model

**Based on the elbow curve and silhouette scores, choose a final K value. Then train a final K-Means model.**

Use `random_state=42` and `n_init=10`.

In [ ]:
FINAL_K = 4

kmeans_final = KMeans(n_clusters=FINAL_K, random_state=42, n_init=10)
kmeans_final.fit(X_scaled)

print(f"Final K-Means model trained with K = {FINAL_K}")
print(f"Final inertia: {kmeans_final.inertia_:.2f}")

**Add the final cluster labels to the original dataframe in a new column called `Cluster`.**

In [ ]:
df['Cluster'] = kmeans_final.labels_
print("Cluster column added successfully.")

**Check the first five rows after adding the cluster labels.**

In [ ]:
df.head()

## Cluster Analysis

Now we need to understand what each cluster means.

**Create a summary table using `groupby()` to show the mean values of each feature for each cluster.**

In [ ]:
cluster_summary = df.groupby('Cluster').mean().round(2)
print("Cluster Summary (mean values per cluster):")
cluster_summary

**Check how many customers are in each cluster.**

In [ ]:
cluster_counts = df['Cluster'].value_counts().sort_index()
print("Number of customers per cluster:")
print(cluster_counts)

plt.figure(figsize=(7, 4))
cluster_counts.plot(kind='bar', color=['steelblue','darkorange','seagreen','tomato'], edgecolor='black')
plt.xlabel('Cluster', fontsize=12)
plt.ylabel('Number of Customers', fontsize=12)
plt.title('Customer Count per Cluster', fontsize=14, fontweight='bold')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Visualizing the Final Clusters

Since the dataset has many features, we will use PCA to reduce the data into two components only for visualization.

This visualization does not replace the original clustering. It only helps us see the clusters in a 2D plot.

Use PCA with 2 components and plot the clusters.

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

explained_var = pca.explained_variance_ratio_
print(f"PCA explained variance: PC1={explained_var[0]:.2%}, PC2={explained_var[1]:.2%}")
print(f"Total explained variance: {sum(explained_var):.2%}")

palette = ['steelblue', 'darkorange', 'seagreen', 'tomato',
           'purple', 'gold', 'crimson', 'teal', 'slateblue']

plt.figure(figsize=(10, 7))
for cluster_id in sorted(df['Cluster'].unique()):
    mask = df['Cluster'] == cluster_id
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1],
                label=f'Cluster {cluster_id}',
                alpha=0.5, s=20,
                color=palette[cluster_id])

plt.xlabel(f'Principal Component 1 ({explained_var[0]:.1%} variance)', fontsize=12)
plt.ylabel(f'Principal Component 2 ({explained_var[1]:.1%} variance)', fontsize=12)
plt.title('Customer Clusters Visualized via PCA (2D Projection)', fontsize=14, fontweight='bold')
plt.legend(title='Cluster', fontsize=10)
plt.tight_layout()
plt.show()

**Output Interpretation**

The PCA plot gives a simplified 2D view of the clusters.

If the clusters are not perfectly separated, that is normal because the original dataset has many features and the plot only shows two compressed dimensions.

## Final Questions

Answer the following questions:

1. Why is this an unsupervised learning problem?

2. Why did we remove the `CUST_ID` column?

3. Which columns had missing values?

4. How did you handle the missing values?

5. Why is scaling important before applying K-Means?

6. Which K value did you choose? Explain your answer using the elbow method and silhouette score.

7. Based on the cluster summary table, describe each customer segment in your own words.

8. Which cluster may represent high-value customers?

9. Which cluster may represent customers who rely more on cash advance?

10. How can a company use these clusters for marketing strategy?

In [ ]:
1. What is unsupervised learning?

   Unsupervised learning is a machine learning approach where the dataset does not contain target labels or predefined outputs. The model analyzes the data independently to discover hidden structures and similarities between customers. In this assignment, K-Means clustering grouped customers based on their spending and credit card behavior patterns without knowing any correct category beforehand.

2. Why was CUST_ID removed?

   The CUST_ID column was removed because it is only a unique identifier and does not represent any customer behavior or financial activity. Keeping it in the dataset could negatively affect the K-Means clustering process since the algorithm relies on distance calculations between feature values.

3. Which columns contained missing values?

   The columns that contained missing values were:

 MINIMUM_PAYMENTS
 CREDIT_LIMIT

4. How were the missing values handled?

   The missing values were handled using mean imputation. Each missing value was replaced with the average value of its corresponding column. This method allowed us to keep all customer records while maintaining the overall distribution of the data.

5. Why was scaling applied before K-Means clustering?

   Feature scaling was necessary because K-Means uses Euclidean distance to measure similarity between data points. Some features, such as CREDIT_LIMIT, have much larger values than others like PURCHASES_FREQUENCY. Without scaling, larger-value features would dominate the clustering process. StandardScaler was used to ensure all features contributed equally.

6. Which value of K was selected and why?

   We selected K = 4 because the elbow method showed that the decrease in inertia started slowing after four clusters, indicating diminishing returns beyond that point. In addition, the silhouette score for K = 4 was among the highest values, suggesting good cluster separation. Four clusters also provided customer groups that were easy to interpret from a business perspective.

7. Describe the customer segments identified by the model.


Cluster 0: Inactive Customers

  Customers in this cluster have low balances and very low purchase activity. They rarely use their credit cards and show limited engagement.

Cluster 1: Cash Advance Users

  This group has high cash advance usage and frequency but relatively low purchase activity. These customers appear to rely more on cash withdrawals than regular spending.

Cluster 2: Active Shoppers

  Customers in this cluster make frequent purchases and actively use their credit cards. They maintain moderate balances and represent regular card users.

Cluster 3: Premium Customers

  This segment includes customers with high balances, large credit limits, and high payment amounts. They appear to be high-value and financially strong customers.

8. Which cluster represents the highest-value customers?

   Cluster 3 represents the highest-value customers because it has the highest average credit limits, balances, and payment amounts. These characteristics indicate strong purchasing power and consistent card usage.

9. Which cluster is associated with cash advance behavior?

   Cluster 1 is strongly associated with cash advance behavior because customers in this segment show significantly higher CASH_ADVANCE values and CASH_ADVANCE_FREQUENCY compared to the other clusters.

10. Suggested marketing strategies for each cluster


Cluster 0: Inactive Customers

  Offer re-engagement campaigns, reward points, and fee waivers to encourage more card usage.

Cluster 1: Cash Advance Users

  Promote personal loan options or lower-interest financial products to reduce reliance on cash advances.

Cluster 2: Active Shoppers

  Provide cashback rewards, shopping discounts, and card upgrade opportunities to increase loyalty.

Cluster 3: Premium Customers

  Offer premium benefits such as travel rewards, airport lounge access, concierge services, and exclusive VIP programs.
